In [ ]:
# ============================================================
# WORKSHEET 4
# FCN for Devnagari Digit Classification using Keras
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical

# ============================================================
# 1. DATA PREPARATION
# ============================================================

# Update these paths if needed
train_dir = "dataset/Train"
test_dir = "dataset/Test"

img_height, img_width = 28, 28
num_classes = 10

def load_images_from_folder(folder):
    images = []
    labels = []

    # Get class folders in sorted order
    class_names = sorted([d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))])
    class_map = {name: i for i, name in enumerate(class_names)}

    print("Class mapping:", class_map)

    for class_name in class_names:
        class_path = os.path.join(folder, class_name)
        label = class_map[class_name]

        for filename in os.listdir(class_path):
            img_path = os.path.join(class_path, filename)

            try:
                img = Image.open(img_path).convert("L")          # grayscale
                img = img.resize((img_width, img_height))       # resize to 28x28
                img = np.array(img, dtype=np.float32) / 255.0   # normalize to [0,1]

                images.append(img)
                labels.append(label)
            except Exception as e:
                print(f"Skipping {img_path} بسبب error: {e}")

    return np.array(images), np.array(labels)

# Load dataset
x_train, y_train = load_images_from_folder(train_dir)
x_test, y_test = load_images_from_folder(test_dir)

print("Original train image shape:", x_train.shape)
print("Original test image shape:", x_test.shape)
print("Original y_train shape:", y_train.shape)
print("Original y_test shape:", y_test.shape)

# Visualize some training images
plt.figure(figsize=(10, 4))
samples_to_show = min(10, len(x_train))
for i in range(samples_to_show):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

# Reshape for Keras input: (samples, 28, 28, 1)
x_train = x_train.reshape(-1, img_height, img_width, 1)
x_test = x_test.reshape(-1, img_height, img_width, 1)

# One-hot encode labels
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("Reshaped x_train:", x_train.shape)
print("Reshaped x_test:", x_test.shape)
print("One-hot y_train:", y_train_cat.shape)
print("One-hot y_test:", y_test_cat.shape)

# ============================================================
# 2. BUILD THE FCN MODEL
# ============================================================

model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="sigmoid"),
    keras.layers.Dense(128, activation="sigmoid"),
    keras.layers.Dense(256, activation="sigmoid"),
    keras.layers.Dense(10, activation="softmax")
])

print("\nMODEL SUMMARY:")
model.summary()

# ============================================================
# 3. COMPILE THE MODEL
# ============================================================

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# 4. TRAIN THE MODEL
# ============================================================

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="best_devnagari_fcn_model.keras",
        save_best_only=True,
        monitor="val_loss"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    )
]

history = model.fit(
    x_train,
    y_train_cat,
    batch_size=128,
    epochs=20,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# 5. VISUALIZE TRAINING AND VALIDATION PERFORMANCE
# ============================================================

train_loss = history.history["loss"]
val_loss = history.history["val_loss"]
train_acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_acc, label="Training Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

# ============================================================
# 6. EVALUATE THE MODEL
# ============================================================

test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=2)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# ============================================================
# 7. SAVE AND LOAD THE MODEL
# ============================================================

model.save("devnagari_fcn_model.h5")
print("\nModel saved as devnagari_fcn_model.h5")

loaded_model = tf.keras.models.load_model("devnagari_fcn_model.h5")
print("Saved model loaded successfully.")

loaded_loss, loaded_acc = loaded_model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Loaded Model Test Loss: {loaded_loss:.4f}")
print(f"Loaded Model Test Accuracy: {loaded_acc:.4f}")

# ============================================================
# 8. MAKE PREDICTIONS
# ============================================================

predictions = loaded_model.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(y_test_cat, axis=1)

print("\nFirst 10 predicted labels:", predicted_labels[:10])
print("First 10 true labels:     ", true_labels[:10])

# ============================================================
# 9. SHOW SOME TEST PREDICTIONS
# ============================================================

plt.figure(figsize=(12, 6))
samples_to_show = min(10, len(x_test))

for i in range(samples_to_show):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.title(f"P:{predicted_labels[i]} / T:{true_labels[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# ============================================================
# 10. SHOW MISCLASSIFIED IMAGES
# ============================================================

misclassified_idx = np.where(predicted_labels != true_labels)[0]
print(f"\nTotal misclassified images: {len(misclassified_idx)}")

if len(misclassified_idx) > 0:
    plt.figure(figsize=(12, 6))
    for i, idx in enumerate(misclassified_idx[:10]):
        plt.subplot(2, 5, i + 1)
        plt.imshow(x_test[idx].reshape(28, 28), cmap="gray")
        plt.title(f"P:{predicted_labels[idx]} / T:{true_labels[idx]}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("All images were correctly classified!")

# ============================================================
# 11. OPTIONAL: SINGLE IMAGE PREDICTION EXAMPLE
# ============================================================

sample_index = 0
sample_image = x_test[sample_index:sample_index + 1]
sample_prediction = loaded_model.predict(sample_image)
sample_label = np.argmax(sample_prediction, axis=1)[0]

print(f"\nPredicted label for first test image: {sample_label}")
print(f"True label for first test image: {true_labels[sample_index]}")